In [1]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files


In [2]:
up = files.upload()

nama_file = list(up.keys())[0]

print("File terbaca:", nama_file)


Saving KUISIONER LATIHAN MK STATISTIKA PROBABILITAS (Responses).xlsx to KUISIONER LATIHAN MK STATISTIKA PROBABILITAS (Responses).xlsx
File terbaca: KUISIONER LATIHAN MK STATISTIKA PROBABILITAS (Responses).xlsx


In [3]:
JUMLAH_KOLOM_DEMOGRAFI = 5

df = pd.read_excel(nama_file)

butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:].copy()

# Mengeluarkan kolom durasi penggunaan gadget
kolom_durasi = [
    kolom for kolom in butir.columns
    if str(kolom).strip().lower().startswith("durasi")
]

print("Kolom yang dikeluarkan:", kolom_durasi)

butir = butir.drop(columns=kolom_durasi)

print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))


Kolom yang dikeluarkan: ['Durasi (jam) rata-rata penggunaan gadget perhari, untuk , kebutuhan harian sampai dengan hiburan']
Jumlah responden : 38
Jumlah butir     : 27
Data kosong      : 0


In [4]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5
}

num = butir.replace(skala).apply(
    pd.to_numeric, errors="coerce"
)

# Harus 0 jika semua jawaban berhasil dikonversi
print(
    "Sel yang gagal dikonversi:",
    int(num.isna().sum().sum())
)

num.head()


Sel yang gagal dikonversi: 0


/tmp/ipykernel_1204/3611101596.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memanfaatkan AI untuk menemukan ide alternatif ketika mengembangkan topik tugas atau projek perkuliahan.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya tidak mengkhawatirkan hasil yang diberikan AI],Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Rsspon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Rsspon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Rsspon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Rsspon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,5,5,5,5,4,4,5,5,5,1,...,4,4,5,1,5,4,1,1,1,5
1,4,4,5,4,3,4,4,3,1,1,...,4,2,2,2,2,4,5,5,5,5
2,2,4,2,4,4,4,2,4,4,4,...,4,2,4,4,2,2,2,4,4,5
3,4,5,5,4,5,5,5,4,4,4,...,2,1,1,1,1,1,5,5,5,5
4,5,4,5,5,4,4,3,3,1,3,...,4,1,5,5,4,5,5,4,4,5


In [5]:
dimensi = {}

for kolom in butir.columns:
    cocok = re.match(r"^(.*?) \[", str(kolom))

    if cocok is None:
        raise ValueError(
            f"Format nama kolom tidak sesuai: {kolom}"
        )

    nama = cocok.group(1).strip().rstrip(".").strip()

    dimensi.setdefault(nama, []).append(kolom)


for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")


Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 6 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik: 4 butir
Rsspon emosional: 4 butir


In [6]:
n = len(num)

t = stats.t.ppf(0.975, n - 2)

r_tabel = t / np.sqrt((n - 2) + t**2)

print(
    f"n = {n} | "
    f"r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}"
)


hasil_validitas = []

for nama, daftar in dimensi.items():

    x = num[daftar]

    total = x.sum(axis=1)

    for i, kolom in enumerate(daftar, 1):

        r, p = stats.pearsonr(x[kolom], total)

        r_koreksi, _ = stats.pearsonr(
            x[kolom],
            total - x[kolom]
        )

        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid"
        })


tabel_validitas = pd.DataFrame(hasil_validitas)

display(tabel_validitas)


n = 38 | r tabel (α=0,05; df=36) = 0.320


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.723,0.547,0.0000,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.393,0.260,0.0145,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.428,0.257,0.0074,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.470,0.240,0.0029,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.703,0.523,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.739,0.627,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.738,0.550,0.0000,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.651,0.516,0.0000,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.750,0.567,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.643,0.430,0.0000,Valid


In [7]:
def cronbach_alpha(x):

    k = x.shape[1]

    return k / (k - 1) * (
        1 - x.var(ddof=1).sum() /
        x.sum(axis=1).var(ddof=1)
    )


def kategori(a):

    return (
        "Sangat baik" if a >= 0.90 else
        "Baik" if a >= 0.80 else
        "Reliabel" if a >= 0.70 else
        "Tidak reliabel"
    )


hasil_reliabilitas = []

alpha_dihapus = []


for nama, daftar in dimensi.items():

    x = num[daftar]

    a = cronbach_alpha(x)

    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a)
    })

    for i, kolom in enumerate(daftar, 1):

        if len(daftar) > 2:

            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(
                    cronbach_alpha(
                        x.drop(columns=kolom)
                    ), 3
                )
            })


tabel_reliabilitas = pd.DataFrame(
    hasil_reliabilitas
)

tabel_alpha_dihapus = pd.DataFrame(
    alpha_dihapus
)


display(tabel_reliabilitas)

display(tabel_alpha_dihapus)


,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.713,Reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,6,0.728,Reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.754,Reliabel
3,Prokrastinasi akademik,4,0.817,Baik
4,Rsspon emosional,4,0.813,Baik


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.645
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.714
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.715
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.728
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.653
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.639
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.644
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.683
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.656
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.701


In [9]:
# Mengambil subset acak 20 responden
num_20 = num.sample(20, random_state=1)

n = len(num_20)

t = stats.t.ppf(0.975, n - 2)

r_tabel_20 = t / np.sqrt((n - 2) + t**2)

print("Jumlah responden:", n)
print("r tabel:", round(r_tabel_20, 3))


# Menguji validitas pada subset 20 responden
hasil_validitas_20 = []

for nama, daftar in dimensi.items():

    x = num_20[daftar]

    total = x.sum(axis=1)

    for i, kolom in enumerate(daftar, 1):

        r, p = stats.pearsonr(x[kolom], total)

        hasil_validitas_20.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Keterangan": "Valid" if r > r_tabel_20 else "Tidak valid"
        })


tabel_validitas_20 = pd.DataFrame(hasil_validitas_20)

display(tabel_validitas_20)


# Menghitung jumlah butir valid dan tidak valid
jumlah_valid = (
    tabel_validitas_20["Keterangan"] == "Valid"
).sum()

jumlah_tidak_valid = (
    tabel_validitas_20["Keterangan"] == "Tidak valid"
).sum()

print("Jumlah butir valid:", jumlah_valid)
print("Jumlah butir tidak valid:", jumlah_tidak_valid)



Jumlah responden: 20
r tabel: 0.444


,Dimensi,Butir,r hitung,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.625,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.573,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.672,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.619,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.679,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.782,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.652,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.832,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.690,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.652,Valid


Jumlah butir valid: 27
Jumlah butir tidak valid: 0


In [ ]:
gabung = tabel_validitas.merge(
    tabel_alpha_dihapus,
    on=["Dimensi", "Butir"],
    how="left"
)


with pd.ExcelWriter(
    "hasil_uji_validitas_reliabilitas.xlsx"
) as w:

    gabung.to_excel(
        w,
        sheet_name="Validitas",
        index=False
    )

    tabel_reliabilitas.to_excel(
        w,
        sheet_name="Reliabilitas",
        index=False
    )


files.download(
    "hasil_uji_validitas_reliabilitas.xlsx"
)
